# Retrieval-Augmented Generation (RAG) Laboratory Exercise



Doing this lab exercise, I have encountered some issues such as the model and package dependencies. The original model, `llama-3.1-8b-instant`, was replaced with `openai/gpt-oss-20b` because the former was deprecated by Groq for the intended setup. The package installation was also adjusted to use specific compatible versions because the original environment setup caused dependency issues when using the latest package releases.


#NOTE: The task 1 to 3 are located at the bottom of this notebook.



## Cell 1: Environment Setup and Package Installation

This cell prepares the Google Colab environment by installing all libraries required for the RAG activity. In the original version, the packages were installed without specific versions, which meant Colab would automatically use the latest available releases. When I worked with the notebook, this caused dependency compatibility problems among the LangChain-related packages.

To make the environment more stable, the updated cell installs fixed versions of `langchain`, `langchain-community`, `langchain-groq`, and `langchain-huggingface`. These packages work together to handle the core RAG process, community integrations, Groq model access, and Hugging Face embeddings.

The remaining packages support other parts of the workflow. `chromadb` is used as the vector database, `pypdf` supports PDF documents, and `unstructured[pdf]` provides additional PDF parsing capability. Overall, this cell makes sure the required environment is ready before the rest of the notebook runs.

In [1]:
# Cell 1: Environment Setup & Package Installation

!pip install -q \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-groq==0.3.8" \
    "langchain-huggingface==0.3.1" \
    chromadb \
    pypdf \
    "unstructured[pdf]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.1 MB/s eta 0:00:00
  

## Cell 2: Secure API Key Configuration

This cell securely configures the Groq API key that will be needed when the language model is initialized later. It starts by importing `os`, which allows the code cell to work with environment variables, and `getpass`, which provides a safer way to enter sensitive information.

The condition checks whether `GROQ_API_KEY` is already available in the current environment. If it is not, `getpass.getpass()` prompts for the API key without displaying the actual characters being entered. The value is then stored in `os.environ["GROQ_API_KEY"]`, allowing the Groq integration to use it automatically.

The final print statement confirms that the API key configuration was completed. I find this approach more appropriate than typing the key directly into the notebook because the actual API key does not appear visibly in the code cell or its output.

In [ ]:
# Cell 2: Secure API Key Configuration

import os
import getpass

# Prompts for API key securely without saving or echoing plain text
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass(
        "Enter your Groq API Key: "
    )

print("Groq API key configured successfully!")

Enter your Groq API Key: ··········
Groq API key configured successfully!


## Cell 3: Loading Custom Data and Chunking

This cell loads the custom documents that will serve as the knowledge source of the RAG chatbot. It first imports `DirectoryLoader`, which reads files from a directory, and `RecursiveCharacterTextSplitter`, which is used to divide longer document contents into smaller chunks.

The `my_data` folder is created using `os.makedirs()`. The option `exist_ok=True` allows the cell to run even if the folder already exists. `DirectoryLoader` then scans the folder using the pattern `**/*.*`, which allows it to find different file types and files located inside subfolders. Calling `loader.load()` stores the loaded documents in `raw_documents`.

The next part prepares the text for retrieval. `RecursiveCharacterTextSplitter` uses a `chunk_size` of 500 and a `chunk_overlap` of 50. This means each document is divided into smaller sections while keeping a small amount of shared text between neighboring chunks. That overlap helps preserve context that might otherwise be separated at a chunk boundary.

Finally, `split_documents()` creates the processed chunks stored in `documents`, and the print statement reports both the number of original documents and the total number of chunks. This output is useful for checking whether the uploaded files were successfully loaded and processed.

In [ ]:
# Cell 3: Loading Custom Data & Chunking

from langchain_community.document_loaders import DirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create target data directory

os.makedirs("my_data", exist_ok=True)


# Load all documents from the directory

loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)

raw_documents = loader.load()


# Split documents into smaller semantic chunks

text_splitter = RecursiveCharacterTextSplitter(

chunk_size=500,

chunk_overlap=50

)

documents = text_splitter.split_documents(raw_documents)


print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 3/3 [00:00<00:00, 46.82it/s]

Loaded 3 raw document(s) and split into 20 chunks.


## Cell 4: Embedding Model and Vector Database Indexing

This cell transforms the document chunks into embeddings and stores them in a Chroma vector database. It imports `HuggingFaceEmbeddings` for creating the numerical representations of the text and `Chroma` for storing and searching those representations.

The embedding model used is `all-MiniLM-L6-v2`. Instead of comparing only exact words, this model converts text into numerical vectors that represent semantic meaning. This makes it possible for the retriever to identify document chunks that are conceptually related to a user's question.

`Chroma.from_documents()` takes the chunks stored in `documents` and generates embeddings for each one using the selected model. The resulting vector database is stored in `vectorstore`.

The last line converts the vector store into a retriever. Setting `k` to 3 means the retriever will return the three chunks it considers most relevant whenever a question is submitted. These retrieved chunks will later become the context used by the language model.

In [ ]:
# Cell 4: Embedding Model & Vector DB Indexing

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma


# Initialize open-source embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# Store embeddings into Chroma vector database

vectorstore = Chroma.from_documents(

documents=documents,

embedding=embeddings

)


# Set vectorstore as a retriever

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Cell 5: Model Initialization and Domain System Prompt

This cell prepares the language model and defines the instructions that guide its responses. `ChatGroq` is imported to access the selected model through Groq, while `ChatPromptTemplate` is used to organize the system message and the user's input.

The original laboratory model, `llama-3.1-8b-instant`, remains visible as a commented line for reference. The active model is now `openai/gpt-oss-20b`, which replaces the deprecated model while keeping the same Groq-based setup. The `temperature` is set to 0 so that the responses are more consistent and focused on the retrieved information rather than being highly variable.

The `system_prompt` establishes the main restriction of the chatbot. It tells the model to answer only from the provided context and to return a specific fallback message when the answer cannot be found. The `{context}` placeholder will later contain the retrieved document chunks.

Finally, `ChatPromptTemplate.from_messages()` combines the system instruction with the user's question represented by `{input}`. This prepares the message structure that will be used when the RAG pipeline is assembled.

In [ ]:
# Cell 5: Model Initialization and Domain System Prompt

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate


# Initialize the SLM

llm = ChatGroq(

#model_name="llama-3.1-8b-instant",
model_name="openai/gpt-oss-20b",

temperature=0

)


# Custom domain system prompt

system_prompt = (

"You are a specialized AI assistant for the user's uploaded domain.\n"

"Answer questions strictly using ONLY the provided context below.\n"

"If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"

"Context:\n{context}"

)


prompt = ChatPromptTemplate.from_messages([

("system", system_prompt),

("human", "{input}"),

])

## Cell 6: Pipeline Assembly

This cell connects the retrieval process with the language model so that the separate components work together as a complete RAG pipeline. It imports `create_retrieval_chain` and `create_stuff_documents_chain`, which are responsible for linking the retrieved document chunks to the prompt and model.

`create_stuff_documents_chain(llm, prompt)` creates `combine_docs_chain`. This component takes the retrieved chunks and places their contents into the `{context}` section of the prompt before sending the completed prompt to the language model.

The next line creates `rag_chain` using `create_retrieval_chain()`. This connects the retriever from the previous cell with `combine_docs_chain`. When a user submits a question, the retriever first searches for relevant chunks, then those chunks are passed to the model as supporting context.

This cell is where the individual RAG components become one complete question-answering workflow.

In [ ]:
# Cell 6: Pipeline Assembly

from langchain.chains import create_retrieval_chain

from langchain.chains.combine_documents import create_stuff_documents_chain


# Combine prompt and LLM to process context

combine_docs_chain = create_stuff_documents_chain(llm, prompt)


# Assemble full retrieval-augmented generation chain

rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Cell 7: Testing the Custom Domain Chatbot

This final cell tests the completed RAG chain by sending a question through the retrieval and generation process. The selected query asks, `"What to do during flooding?"`

The question is stored in `user_query` and passed to `rag_chain.invoke()` using the `input` key. The returned result is saved in `response`, which contains both the generated answer and the chunks retrieved from the vector database.

`response["answer"]` displays the model's final response. If the uploaded domain documents do not contain information about the Moon landing, this question also works as a useful out-of-domain test. Based on the system prompt, the model should avoid answering from unrelated outside knowledge and instead use the specified fallback response when the information is not available in the retrieved context.

The last loop goes through `response["context"]` and prints the source of each retrieved chunk. This makes it possible to observe which document files were selected by the retriever and provides a simple way to inspect the source information behind the model's response.

In [ ]:
# Cell 7: Testing Your Custom Domain Chatbot


# Test Case 1: In-Domain Query

user_query = "What to do during flooding?"

response = rag_chain.invoke({"input": user_query})


print("--- DOMAIN QUERY ANSWER ---")

print(response["answer"])


print("\n--- RETRIEVED SOURCE CHUNKS ---")

for i, doc in enumerate(response["context"]):

    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
**During a flood:**

- **Avoid** walking or driving through flooded areas.  
- **Avoid** places that flood quickly or frequently experience flash floods.  
- **Secure** your home before evacuating: turn off the main electrical switch and close windows and doors if it is safe to do so.  
- **Evacuate** only when authorities announce it.  
- **Do not leave pets restrained**; release them or take them with you during evacuation.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/pasig_flood_and_typhoon_preparedness.txt
Chunk 2 Source: my_data/pasig_flood_and_typhoon_preparedness.txt
Chunk 3 Source: my_data/pasig_flood_and_typhoon_preparedness.txt


##Practical Experiments & Deliverables

##Task 1: Domain & Dataset Description
Chosen Domain

The chosen domain for this RAG chatbot is Disaster Preparedness and Emergency Information for Pasig City. The dataset focuses on practical information that residents may need before or during emergency situations, particularly floods, typhoons, fires, and earthquakes. It also contains Pasig City barangay emergency contact numbers and evacuation center locations.

Problem Statement

During emergencies and disasters, people need to access important information quickly, such as what actions to take, whom to contact, and where to evacuate. However, this information may be spread across different sources, making it harder to find the specific information needed at the moment.

This RAG chatbot is designed to retrieve relevant information from a small collection of Pasig City disaster preparedness and emergency documents. It allows a user to ask questions about emergency procedures, preparedness measures, barangay emergency hotlines, and evacuation center locations. The chatbot is instructed to answer using only the information available in the uploaded domain documents and to avoid providing an unsupported answer when the requested information is not found.

Source Files Uploaded to ./my_data/

The following source files are used as the knowledge base for the RAG chatbot:

pasig_fire_and_earthquake_preparedness.txt
Contains earthquake safety procedures, fire prevention tips, cooking-oil or grease fire safety, and the PASS method for using a fire extinguisher.
pasig_flood_and_typhoon_preparedness.txt
Contains preparedness guidance for before, during, and after floods and typhoons, along with recommended emergency go-bag contents.
pasig_barangay_emergency_hotlines_and_evacuation centers.txt
Contains emergency contact numbers for different Pasig City barangays and listed evacuation center locations.

Together, these three text files provide the domain-specific information that will be loaded from the ./my_data/ directory, divided into smaller chunks, converted into embeddings, and retrieved by the RAG chatbot when answering relevant user questions.

### Task 2: Out-of-Domain Fallback Test

To test whether the chatbot follows the domain restriction defined in the system prompt, I asked a question that is completely unrelated to the uploaded Pasig City disaster preparedness and emergency information dataset.

### Test Query

**Question:**
`Who was the first person to walk on the Moon, and in what year?`

This question was intentionally selected because the uploaded dataset focuses on disaster preparedness, emergency procedures, barangay emergency hotlines, and evacuation centers in Pasig City. It does not contain information about space exploration or the Moon landing.

### Output

`I cannot answer based on the provided domain data.`

### Verification

The chatbot correctly returned the expected fallback message instead of answering the question using the language model's general knowledge. This shows that the domain instruction is working as intended for this test. Although the language model may already have general knowledge about the Moon landing, it followed the system prompt and limited its response to information available from the retrieved domain context.

Therefore, the out-of-domain fallback test was **successful**.


In [ ]:
# Task 2: Out-of-Domain Fallback Test


user_query = "Who was the first person to walk on the Moon, and in what year?"

response = rag_chain.invoke({"input": user_query})


print("--- DOMAIN QUERY ANSWER ---")

print(response["answer"])


print("\n--- RETRIEVED SOURCE CHUNKS ---")

for i, doc in enumerate(response["context"]):

    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
I cannot answer based on the provided domain data.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt
Chunk 2 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt
Chunk 3 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt


## Task 3: Hallucination Stress-Test

For this task, I intentionally weakened the grounding controls of the chatbot to observe how the response would change.

### Step 1: Changed Parameters

In Cell 5, I changed:

`temperature=0`

to:

`temperature=1`

I also removed the instruction:

`If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'`

This means the model was given more freedom in generating its response and was no longer explicitly instructed to return the fallback message when the answer was missing from the retrieved context.



### Step 2: Re-ran Query

I used the exact same question from Task 2:

**Question:**
`Who was the first person to walk on the Moon, and in what year?`

Output:
`The first person to walk on the Moon was Neil Armstrong, and he did so in 1969 (Apollo 11 mission).`

### Step 3: Observation and Compare

The result was different from the baseline test in `Task 2`. With temperature=0 and the fallback instruction enabled, the chatbot responded with `"I cannot answer based on the provided domain data."` This showed that the original configuration remained grounded when the requested information was not available in the dataset.

After changing the temperature to `1.0` and removing the explicit fallback instruction, the chatbot answered the Moon-landing question even though the retrieved Pasig City disaster preparedness documents do not contain information about Neil Armstrong, Apollo 11, or space exploration.

Although the generated answer is generally known to be factually correct, it is unsupported by the retrieved context used by this RAG system. For this experiment, this shows that the modified configuration became less grounded and introduced information outside the provided domain data.

The comparison also shows that increasing the temperature does not automatically cause hallucination by itself. However, together with removing the explicit fallback rule, the modified configuration provided less control over how the model handled information that was missing from the retrieved context. In this test, the original configuration was more reliable for keeping the chatbot within the boundaries of the uploaded dataset.

In [ ]:
# Task 3 Hallucination Stress-Test

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate


# Initialize the SLM

llm = ChatGroq(

#model_name="llama-3.1-8b-instant",
model_name="openai/gpt-oss-20b",

temperature=1

)


# Custom domain system prompt

system_prompt = (

"You are a specialized AI assistant for the user's uploaded domain.\n"

"Answer questions strictly using ONLY the provided context below.\n"


"Context:\n{context}"

)


prompt = ChatPromptTemplate.from_messages([

("system", system_prompt),

("human", "{input}"),

])

In [ ]:
# Task 3: Hallucination Stress-Test


user_query = "Who was the first person to walk on the Moon, and in what year?"

response = rag_chain.invoke({"input": user_query})


print("--- DOMAIN QUERY ANSWER ---")

print(response["answer"])


print("\n--- RETRIEVED SOURCE CHUNKS ---")

for i, doc in enumerate(response["context"]):

    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
The first person to walk on the Moon was **Neil Armstrong**, and he did so in **1969** (Apollo 11 mission).

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt
Chunk 2 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt
Chunk 3 Source: my_data/pasig_barangay_emergency_hotlines_and_evacuation centers.txt
